In [6]:
!pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 21.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 43.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.6 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 KB 25.1 MB/s eta 0:00:00


In [9]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from functions2 import train_with_logging, compute_scores
import os

# === Load data ===
df = pd.read_parquet("data/measuring-hate-speech.parquet")
df = df.dropna(subset=['text'])

numerical_cols = ['sentiment', 'respect', 'insult', 'humiliate', 'status',
                  'dehumanize', 'attack_defend', 'hatespeech']
binary_cols = ['target_race', 'target_religion', 'target_origin', 'target_gender',
               'target_sexuality']

df[binary_cols] = df[binary_cols].astype(int)

# === Split ===
train_texts, temp_texts, train_y_num, temp_y_num, train_y_bin, temp_y_bin = train_test_split(
    df['text'], df[numerical_cols], df[binary_cols], test_size=0.3, random_state=42
)
val_texts, test_texts, val_y_num, test_y_num, val_y_bin, test_y_bin = train_test_split(
    temp_texts, temp_y_num, temp_y_bin, test_size=1/3, random_state=42
)

# === Tokenizer ===
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class HateSpeechDataset(torch.utils.data.Dataset):
    def __init__(self, texts, targets_num, targets_bin, tokenizer, max_len=128):
        self.texts = list(texts)
        self.targets_num = torch.tensor(targets_num.values, dtype=torch.float)
        self.targets_bin = torch.tensor(targets_bin.values, dtype=torch.float)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'num_targets': self.targets_num[idx],
            'bin_targets': self.targets_bin[idx]
        }

# === Dataloaders ===
train_loader = DataLoader(HateSpeechDataset(train_texts, train_y_num, train_y_bin, tokenizer), batch_size=16, shuffle=True)
val_loader = DataLoader(HateSpeechDataset(val_texts, val_y_num, val_y_bin, tokenizer), batch_size=16)
test_loader = DataLoader(HateSpeechDataset(test_texts, test_y_num, test_y_bin, tokenizer), batch_size=16)

# === Model ===
class BERTMultiTaskModel(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", num_outputs=8, bin_outputs=5):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_outputs)
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, bin_outputs),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.regressor(pooled), self.classifier(pooled)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERTMultiTaskModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

loss_fn_num = nn.MSELoss()
loss_fn_bin = nn.BCELoss()

In [10]:
# === Training ===

if os.path.exists("checkpoint2.pt"):
    model.load_state_dict(torch.load("checkpoint2.pt"))
    print("✅ Loaded model from checkpoint2.pt")


train_with_logging(
    model, optimizer,
    train_loader, val_loader,
    loss_fn_num, loss_fn_bin,
    device,
    checkpoint_path="checkpoint2.pt",
    num_extra_epochs=8,
    log_file=None,
    plot_graph=False,
    scheduler=scheduler,
    patience=3,
    save_best_model=True
)

# === Evaluation ===
print("\n Final Evaluation on TEST set:")
compute_scores(model, test_loader, device)

KeyboardInterrupt: 

run2 1:40

In [11]:
checkpoint = torch.load("checkpoint2.pt", map_location=device)
print(checkpoint.keys())  # Check what keys are inside


odict_keys(['bert.embeddings.word_embeddings.weight', 'bert.embeddings.position_embeddings.weight', 'bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.transformer.layer.0.attention.q_lin.weight', 'bert.transformer.layer.0.attention.q_lin.bias', 'bert.transformer.layer.0.attention.k_lin.weight', 'bert.transformer.layer.0.attention.k_lin.bias', 'bert.transformer.layer.0.attention.v_lin.weight', 'bert.transformer.layer.0.attention.v_lin.bias', 'bert.transformer.layer.0.attention.out_lin.weight', 'bert.transformer.layer.0.attention.out_lin.bias', 'bert.transformer.layer.0.sa_layer_norm.weight', 'bert.transformer.layer.0.sa_layer_norm.bias', 'bert.transformer.layer.0.ffn.lin1.weight', 'bert.transformer.layer.0.ffn.lin1.bias', 'bert.transformer.layer.0.ffn.lin2.weight', 'bert.transformer.layer.0.ffn.lin2.bias', 'bert.transformer.layer.0.output_layer_norm.weight', 'bert.transformer.layer.0.output_layer_norm.bias', 'bert.transformer.layer.1.attention.q_lin.weight', 'be

In [12]:
#without the training
# === Load the model ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERTMultiTaskModel().to(device)

checkpoint = torch.load("checkpoint2.pt", map_location=device)
model.load_state_dict(checkpoint)

# (optional) Save model if you want
torch.save(model.state_dict(), "model2_loaded.pth")

model.eval()

print("✅ Model loaded successfully and ready for evaluation or prediction!")


✅ Model loaded successfully and ready for evaluation or prediction!


In [13]:
sentences = [
    "I love all people, no matter where they come from.",
    "That group is disgusting and should not exist.",
    "We should build a more inclusive and respectful community.",
    "You don't belong here. Go back to your country."
]

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
encodings = tokenizer(sentences, truncation=True, padding=True, max_length=128, return_tensors="pt")

model.eval()
input_ids = encodings['input_ids'].to(device)
attention_mask = encodings['attention_mask'].to(device)

with torch.no_grad():
    preds_num, preds_bin = model(input_ids=input_ids, attention_mask=attention_mask)

preds_num = preds_num.cpu().numpy()
preds_bin = preds_bin.cpu().numpy()

for idx, sentence in enumerate(sentences):
    print(f"Sentence: {sentence}")
    print(f"Numerical predictions: {preds_num[idx]}")
    print(f"Binary predictions: {preds_bin[idx]}")
    print()


Sentence: I love all people, no matter where they come from.
Numerical predictions: [0.6368833  0.6941257  0.55947834 0.453081   1.7052419  0.5089758
 1.0670835  0.02483576]
Binary predictions: [0.35523498 0.2589233  0.69982755 0.14827095 0.10363307]

Sentence: That group is disgusting and should not exist.
Numerical predictions: [3.574609   3.422676   3.0756555  2.6393573  2.9860637  2.2782896
 2.8115137  0.69246453]
Binary predictions: [0.5497439  0.00562938 0.02655231 0.09151145 0.3118188 ]

Sentence: We should build a more inclusive and respectful community.
Numerical predictions: [ 0.3466357   0.2687346   0.24350673  0.18927436  1.944749    0.28197324
  0.9066525  -0.01280466]
Binary predictions: [0.066765   0.21807507 0.08598721 0.51943004 0.24552318]

Sentence: You don't belong here. Go back to your country.
Numerical predictions: [3.8283768 3.7935395 3.505861  3.0549178 3.3294084 2.1788697 3.3982081
 0.5011611]
Binary predictions: [0.18656547 0.02135676 0.9142482  0.01014285 0.

In [14]:
model.eval()

BERTMultiTaskModel(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Lin